In [14]:
## Test of real photos of hands to see how many we can get right

import torch
import torchvision
import torch
from PIL import Image, ImageOps
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
import torch
from torch import nn
import cv2
import numpy as np
from torchvision import models, transforms

In [21]:
## Dataset with photo in it
data_folder = Path(r"C:\Users\hartmawa\Music\Deep Learning\src\Project\Test_Set")


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

ranks = ["0","A", "2", "3", "4", "5", "6", "7", "8", "9", "10"]

images = []
labels = []





In [32]:
## Go through each image and crop/classyfiy cards

#Initialize Correct Counter
dealer_correct=0
player_correct=0
both_correct=0


#Loop through every image in the folder

for path in sorted(data_folder.iterdir()):
    if path.name == "desktop.ini":
        continue
    
#Crop names to just get player and dealer totals
    name=path.stem.split(" (")[0] #Remove the end of names that are duplicates
    dealer_rank = name.split("-")[0]
    player_rank = name.split("-")[1]
    
    
    
    #convert to integers
    dealer_rank=int(dealer_rank)
    player_rank=int(player_rank)
    
    
    with Image.open(path) as photo:
        image = np.array(
            ImageOps.exif_transpose(photo).convert("RGB")
        )

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (1, 1), 0)
    _, mask = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # Actualy crop the card from the image (Expand to multiple this time)
    card_images=[] #We will put each card into this array
    card_owners=[] #Use this to keep track of dealer vs player card

    split_y=image.shape[0]/2 #Split image in half vertically

    width, height = 224, 320 #Roughly matches actual card aspect ratio
    target = np.float32([[0, 0], [width - 1, 0],[width - 1, height - 1], [0, height - 1]])

    min_area=.01*image.shape[0]*image.shape[1] #Find all of the small contours so that we can ignore them


    for card_contour in contours:
        if cv2.contourArea(card_contour)<min_area:
            continue #only go on if contour is big enough to actaully be a card
    #Fit rotated rectangle around the cards
        rect = cv2.minAreaRect(card_contour)
        points = cv2.boxPoints(rect)
        center = points.mean(axis=0)
        angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
        points = points[np.argsort(angles)]

        # Start on a short edge so the output card is portrait, not sideways.
        side_lengths = np.linalg.norm(points - np.roll(points, -1, axis=0), axis=1)
        if side_lengths[0] + side_lengths[2] > side_lengths[1] + side_lengths[3]:
            points = np.roll(points, -1, axis=0)


        #Map those corners to an upright rectangle and crop from the color photo.


        matrix = cv2.getPerspectiveTransform(points, target)
        crop = cv2.warpPerspective(image, matrix, (width, height))
        card_images.append(Image.fromarray(crop))
        card_owners.append("Dealer" if center[1]<split_y else "Player")

    number_found=len(card_images)
    
    #model_path = Path.home() / "models" / "resnet18_realcard_combinedface.pth"
    #model_path = Path.home() / "models" / "resnet18_grayscale_combinedface.pth"
    model_path = Path.home() / "models" / "resnet18_realcard_androom_combinedface.pth"
    #model_path = Path.home() / "models" / "resnet18_realcard_withfacedown_androom_combinedface.pth"


    values=torch.load(model_path,map_location="cpu",weights_only=True)

    #Built the resnest18 with my trained wieghts
    ranks = values["ranks"]
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(ranks))
    model.load_state_dict(values["model_state_dict"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    #Transfroms to match how we trained
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3), #Make grayscale
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    model.eval() #Set model to evaliu
    player_hand=[]
    dealer_hand=[]
    player_value=0 #Initiate player value
    dealer_value=0

    with torch.no_grad():
        for card_image, owner in zip(card_images, card_owners):
            inputs = transform(card_image).unsqueeze(0).to(device)
            prediction = model(inputs).argmax(dim=1).item()
            rank=ranks[prediction]
            
            #Add to either the player or the dealer total depending on classification made earlier
            if owner=="Dealer":
                dealer_hand.append(rank)
            else:
                player_hand.append(rank)



        

    def hand_total(hand): #Hand counting method for each case
        total=sum(1 if rank =="A" else int(rank) for rank in hand)#sum all cards in player hand. Assume ace is 1 for now

        if "A" in hand and total+10<=21:
            total+=10 #Make ace 11 if we are able to without busting
        return total

    #Add up each hand
    player_value=hand_total(player_hand)
    dealer_value=hand_total(dealer_hand)
    
    
    
    ## Add to totals to see how correct we are
    if player_value == player_rank:
        player_correct+=1
    
    if dealer_value == dealer_rank:
        dealer_correct+=1
        
    if player_value == player_rank and dealer_value == dealer_rank:
        both_correct+=1
        



photo_count = len(list(data_folder.iterdir()))

print(f"Dealer accuracy: {dealer_correct / photo_count:.1%}")
print(f"Player accuracy: {player_correct / photo_count:.1%}")
print(f"Both correct: {both_correct / photo_count:.1%}")


    


Dealer accuracy: 82.5%
Player accuracy: 71.1%
Both correct: 61.9%


In [33]:
## Now we need classyfiy the hit vs stand logic


## Go through each image and check hit/stand accuracy

import pandas as pd

#Load in lookup table
strategy=pd.read_excel(
    "DeepLearningLookupTables.xlsx",
    sheet_name="Total Dependent - Hard"
).set_index("PlayerCard")

#Use hit/stand option if we cannot double or surrender
hit_stand_actions={
    "H":"Hit",
    "S":"Stand",
    "Dh":"Hit",
    "Ds":"Stand",
    "Rh":"Hit",
    "Rs":"Stand"
}


def get_hit_stand(player_total,dealer_card):
  
    #Hard totals from 2 through 4 should hit
    if 2<=player_total<=4:
        return "Hit"

    #Keep the rest of your function here
    #Change dealer ace to match excel column. Assumes filenames use 11 for ace
    if dealer_card==11:
        dealer_card="A"

    #Check that we have a row and column for these values
    if player_total not in strategy.index:
        return None

    if dealer_card not in strategy.columns:
        return None

    #Find action in lookup table and convert to hit or stand
    action_code=str(strategy.loc[player_total,dealer_card]).strip()
    return hit_stand_actions.get(action_code)


#Initialize correct counter
action_correct=0
action_photo_count=0

model.eval() #Set model to evaluation mode


#Loop through every image in the folder
for path in sorted(data_folder.iterdir()):
    if path.name=="desktop.ini":
        continue

    #Crop names to just get player and dealer totals
    name=path.stem.split(" (")[0] #Remove the end of names that are duplicates
    actual_dealer,actual_player=map(int,name.split("-"))

    #Find what the action should be based on the actual totals
    expected_action=get_hit_stand(actual_player,actual_dealer)

    #Stop if the actual totals do not have an action in the table
    if expected_action is None:
        raise ValueError(f"No expected Hit/Stand action for {path.name}")

    #Open photo and convert to numpy array
    with Image.open(path) as photo:
        image=np.array(ImageOps.exif_transpose(photo).convert("RGB"))

    #Attempt to separate cards from table
    gray=cv2.cvtColor(image,cv2.COLOR_RGB2GRAY)
    gray=cv2.GaussianBlur(gray,(1,1),0)
    _,mask=cv2.threshold(gray,120,255,cv2.THRESH_BINARY)

    #Find outlines of cards
    contours,_=cv2.findContours(
        mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE
    )

    split_y=image.shape[0]/2 #Top half is dealer and bottom half is player
    min_area=.01*image.shape[0]*image.shape[1] #Ignore small contours

    width,height=224,320 #Roughly matches actual card aspect ratio
    target=np.float32([
        [0,0],
        [width-1,0],
        [width-1,height-1],
        [0,height-1]
    ])

    #Reset hands for each photo
    predicted_player_hand=[]
    predicted_dealer_hand=[]

    with torch.no_grad():
        for card_contour in contours:
            if cv2.contourArea(card_contour)<min_area:
                continue #Only go on if contour is big enough to be a card

            #Fit rotated rectangle around the card
            rect=cv2.minAreaRect(card_contour)
            points=cv2.boxPoints(rect)
            center=points.mean(axis=0)

            angles=np.arctan2(
                points[:,1]-center[1],
                points[:,0]-center[0]
            )
            points=points[np.argsort(angles)]

            #Start on a short edge so the card is upright instead of sideways
            side_lengths=np.linalg.norm(
                points-np.roll(points,-1,axis=0),
                axis=1
            )

            if side_lengths[0]+side_lengths[2]>side_lengths[1]+side_lengths[3]:
                points=np.roll(points,-1,axis=0)

            #Map corners to an upright rectangle and crop the card
            matrix=cv2.getPerspectiveTransform(points,target)
            crop=cv2.warpPerspective(image,matrix,(width,height))
            card_image=Image.fromarray(crop)

            #Run cropped card through our model
            inputs=transform(card_image).unsqueeze(0).to(device)
            prediction=model(inputs).argmax(dim=1).item()
            rank=ranks[prediction]

            #Add card to either player or dealer hand
            if center[1]<split_y:
                predicted_dealer_hand.append(rank)
            else:
                predicted_player_hand.append(rank)

    #Find action using our predicted cards
    predicted_action=None

    #Need player cards and one dealer card to make a recommendation
    if predicted_player_hand and len(predicted_dealer_hand)==1:
        predicted_player_total=hand_total(predicted_player_hand)
        predicted_dealer_card=hand_total(predicted_dealer_hand)

        predicted_action=get_hit_stand(
            predicted_player_total,predicted_dealer_card
        )

        #Check why we did not get a recommendation
    if predicted_action is None:
        print(f"\nPhoto: {path.name}")
        print("Player cards:",predicted_player_hand)
        print("Dealer cards:",predicted_dealer_hand)

        if not predicted_player_hand:
            print("Reason: No player cards found")

        elif len(predicted_dealer_hand)!=1:
            print("Reason: Expected 1 dealer card, found",
                  len(predicted_dealer_hand))

        else:
            player_total=hand_total(predicted_player_hand)
            dealer_card=hand_total(predicted_dealer_hand)
            dealer_column="A" if dealer_card==11 else dealer_card

            print("Predicted player total:",player_total)
            print("Predicted dealer card:",dealer_card)

            if player_total not in strategy.index:
                print("Reason: Player total not in table")

            elif dealer_column not in strategy.columns:
                print("Reason: Dealer card not in table")

            else:
                print("Action code in table:",
                      repr(strategy.loc[player_total,dealer_column]))
    #Add to correct counter if the actions match
    correct=predicted_action==expected_action

    if correct:
        action_correct+=1

    action_photo_count+=1 #Count each photo tested, including failed predictions

    #Print results for this photo
    print(
        f"{path.name}: "
        f"Expected {expected_action}, "
        f"Predicted {predicted_action or 'No recommendation'} "
        f"— {'Correct' if correct else 'Incorrect'}"
    )


#Print overall accuracy
print(f"\nHard-hand hit/stand accuracy: {action_correct/action_photo_count:.1%}")
print(f"Correct: {action_correct}/{action_photo_count}")

10-11.jpg: Expected Hit, Predicted Stand — Incorrect
10-12.jpg: Expected Hit, Predicted Hit — Correct
10-14 (2).jpg: Expected Hit, Predicted Hit — Correct
10-14 (3).jpg: Expected Hit, Predicted Hit — Correct
10-15 (2).jpg: Expected Hit, Predicted Stand — Incorrect
10-15 (3).jpg: Expected Hit, Predicted Stand — Incorrect
10-15 (4).jpg: Expected Hit, Predicted Hit — Correct
10-15 (5).jpg: Expected Hit, Predicted Hit — Correct
10-15.jpg: Expected Hit, Predicted Hit — Correct
10-16 (2).jpg: Expected Hit, Predicted Stand — Incorrect
10-16 (3).jpg: Expected Hit, Predicted Hit — Correct
10-16 (4).jpg: Expected Hit, Predicted Hit — Correct
10-16.jpg: Expected Hit, Predicted Hit — Correct
10-18 (2).jpg: Expected Stand, Predicted Stand — Correct
10-18.jpg: Expected Stand, Predicted Stand — Correct
10-19.jpg: Expected Stand, Predicted Stand — Correct
10-20 (2).jpg: Expected Stand, Predicted Stand — Correct
10-20 (3).jpg: Expected Stand, Predicted Stand — Correct
10-20 (4).jpg: Expected Stand, Pre